# Grafo de conhecimento combinado

Um caso de `sample/cases.csv` passa por peças das cinco estratégias de `src/`, cada uma usada onde é mais forte:

| etapa | estratégia | peça |
|---|---|---|
| sentenças, negação, limpeza de rótulo | stopwords | `split_sentences`, `detect_polarity`, `clean_label` (GUARDED_LABEL + NLTK) |
| tokens, medidas, referências de figura | tokenizacao | `ClinicalRegexTokenizer` |
| fronteira dos rótulos | sintagmas | HMM + `chunk_nps` |
| menções, siglas, deduplicação, relações | normalizacao | extratores, `GraphBuilder`, `build_relations` |
| sítios anatômicos e conceitos MeSH | dicionarios | gazetteers + `greedy_match` / `link_label_to_concept` |

O resultado são as tabelas de nós e arestas do contrato comum, gravadas em `data/processed/`.

In [ ]:
import sys
from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = next(p for base in (Path.cwd(), *Path.cwd().parents)
            for p in (base, base / "project1") if (p / "src" / "normalizacao").is_dir())
if str(ROOT / "pipelines") not in sys.path:
    sys.path.insert(0, str(ROOT / "pipelines"))
from combinado import carregar_recursos, processar_caso, validar_grafo, rotulo_final
from normalizacao.case_reader import read_case
from normalizacao.core import models as nmodels
from normalizacao.extractors.common import ExtractedEntity
from normalizacao.extractors.findings import ExtractedFinding
from normalizacao.normalizers.case import CaseNormalizer
import pos_bio

recursos = carregar_recursos()

def mostrar(df, n=15, largura=70):
    print(df.head(n).to_string(index=False, max_colwidth=largura))
    if len(df) > n:
        print(f"… (+{len(df) - n} linhas)")

def curto(texto, n=90):
    texto = " ".join(str(texto).split())
    return texto if len(texto) <= n else texto[:n - 1] + "…"


## 1. Caso
Lido de `sample/cases.csv` com o leitor de normalizacao (valida colunas e converte a idade para `Decimal`).

In [ ]:
CASE_ID = "PMC5137649_01"

caso = read_case(ROOT / "sample" / "cases.csv", CASE_ID)
texto = caso.case_text

print(f"case_id={caso.case_id}  article_id={caso.article_id}  age={caso.age}  gender={caso.gender}")
print(f"{len(texto)} caracteres; início do texto:\n")
print(texto[:700] + ("…" if len(texto) > 700 else ""))

resultado = processar_caso(caso, recursos, inspecionar=True)
final = resultado.grafo
dados = resultado.inspecao


## 2. Sentenças — *stopwords*
`split_sentences` só corta em `.` que não precede dígito (`12,476.5` fica inteiro) e preserva os offsets.
As sentenças dão o escopo da negação e das ligações `LOCATED_IN`.

In [ ]:
sentencas = dados["sentencas"]
df_sentencas = pd.DataFrame(
    [{"idx": s.index, "start": s.start, "end": s.end, "sentença": curto(s.text, 100)} for s in sentencas]
)
print(f"{len(sentencas)} sentenças")
mostrar(df_sentencas, n=10, largura=100)

## 3. Tokens — *tokenizacao*
Tokenizador de regex clínico: número e unidade coladas saem separados (`12,476.5` + `ng/ml`) e `(Fig 1)` vira um token só.
Os tokens alimentam o POS tagger, as medidas dos achados e a rejeição de doses falsas.

In [ ]:
tokens = dados["tokens"]
print(f"{len(tokens)} tokens por tipo:", dict(Counter(t.kind for t in tokens).most_common()))
df_tokens = pd.DataFrame([{"index": t.index, "text": t.text, "kind": t.kind, "start": t.start, "end": t.end} for t in tokens])
print("\nTokens que não são palavra nem pontuação:")
mostrar(df_tokens[~df_tokens.kind.isin(["WORD", "PUNCTUATION"])], n=20)

## 4. POS e sintagmas nominais — *sintagmas*
Os tokens acima viram `pos_bio.Token` (unidade marcada com `role=UNIT`, referência de figura forçada a pontuação).
O HMM etiqueta, as regras de reparo corrigem e `chunk_nps` devolve os sintagmas `DET? (NUM UNIT?)? MOD* NOUN+`,
que definem onde começa e termina cada rótulo.

In [ ]:
ptokens, chunks = dados["ptokens"], dados["chunks"]
tipo_lexico, reparos = dados["tipo_lexico"], dados["reparos"]
np_por_token = {t.index: ch for ch in chunks for t in ch}
print("POS da primeira sentença:")
print(" ".join(f"{t.text}/{t.tag}" for t in ptokens if t.start < sentencas[0].end))
print(f"\nEtiquetas: {dict(Counter(t.tag for t in ptokens).most_common())}")
print(f"Reparos aplicados: {[(r[0], r[1]) for r in reparos]}")

df_nps = pd.DataFrame(
    [
        {
            "sintagma": pos_bio.surface(ch),
            "start": ch[0].start,
            "end": ch[-1].end,
            "tipo_léxico": tipo_lexico.get((ch[0].index, ch[-1].index + 1), ""),
        }
        for ch in chunks
    ]
)
print(f"\n{len(chunks)} sintagmas nominais, {int((df_nps['tipo_léxico'] != '').sum())} com tipo no léxico:")
mostrar(df_nps, n=20)

verbos = Counter(t.text.lower() for t in ptokens if t.tag == "VERB" and t.index not in np_por_token)
print("\nVerbos fora de sintagma (candidatos a gatilho):", dict(verbos.most_common(12)))

## 5. Siglas do caso — *normalizacao*
Definições `termo por extenso (SIGLA)` valem só dentro deste caso; o `GraphBuilder` as expande ao normalizar os rótulos.

In [ ]:
siglas = CaseNormalizer(texto).acronym_definitions
print(f"{len(siglas)} siglas definidas no texto")
mostrar(pd.DataFrame([{"sigla": k, "expansão": v} for k, v in siglas.items()]))

## 6. Menções brutas — *normalizacao*
Os extratores de normalizacao rodam num grafo rascunho, na mesma ordem de `normalizacao/pipeline.py`.
O rascunho guarda o rótulo bruto de cada nó para que a menção possa ser localizada no texto depois.

In [ ]:
brutas = dados["brutas"]
CAMPOS = ("symptoms", "histories", "exams", "findings", "diagnoses", "medications", "treatments", "outcomes")


def entidade(item):
    return item.entity if isinstance(item, ExtractedFinding) else item


linhas = []
for campo in CAMPOS:
    for item in getattr(brutas, campo):
        ent = entidade(item)
        linhas.append({"node_id": ent.node.node_id, "type": ent.node.type, "label": ent.node.label,
                       "attributes": nmodels.serialize_attributes(ent.node.attributes), "trigger": ent.trigger,
                       "char_start": ent.char_start, "char_end": ent.char_end})
for r in brutas.exam_results:
    linhas.append({"node_id": r.node.node_id, "type": "ExamResult", "label": r.node.label,
                   "attributes": nmodels.serialize_attributes(r.node.attributes), "trigger": f"exame {r.exam_node.label}",
                   "char_start": r.char_start, "char_end": r.char_end})
df_brutas = pd.DataFrame(linhas, columns=["node_id", "type", "label", "attributes", "trigger", "char_start", "char_end"])

print(f"{len(df_brutas)} menções brutas")
print(df_brutas["type"].value_counts().to_dict())
mostrar(df_brutas[["node_id", "type", "label", "attributes", "trigger"]], n=40, largura=55)

## 7. Refinamento das menções
Cada menção é localizada no texto e passa por quatro correções, cada uma vinda de uma estratégia:

1. **tokenizacao** — corta o rótulo antes de um token `FIGURE_REF` e o nome de exame laboratorial na pontuação
   (`0-0.04 ng/ml), creatine kinase` → `creatine kinase`); recalcula o tamanho do achado com a sequência de tokens
   `N UNIT (x N UNIT)*`; descarta `Medication` cuja "dose" é parte de um token de concentração (`6iu/ml`).
2. **sintagmas** — o rótulo de `Finding` vira o sintagma nominal do núcleo; em `Symptom`/`History`/`Diagnosis`,
   só quando o rótulo começa fora de um sintagma (`and bleeding`, `performed and no …`). Artigo, número e unidade saem.
3. **stopwords** — negação detectada na oração que antecede a menção (só `present → absent`).
4. **stopwords** — `clean_label` com a lista do NLTK e a lista de proteção, aplicado só ao rótulo (GUARDED_LABEL).

In [ ]:
registros, exames_de_resultado = dados["registros"], dados["exames_de_resultado"]
normalizador = CaseNormalizer(texto)
linhas = [
    {"node_id": reg["bruta"].node.node_id, "type": reg["tipo"], "antes": reg["bruta"].node.label,
     "depois": "—" if reg["descartar"] else normalizador.normalize_entity_label(reg["rotulo"]), "estratégia": est, "correção": nota}
    for reg in registros + [{"bruta": ExtractedEntity(r.exam_node, "", "", 0, 0), **ref} for r, ref in exames_de_resultado]
    for est, nota in reg["notas"]
]
df_refino = pd.DataFrame(linhas, columns=["node_id", "type", "antes", "depois", "estratégia", "correção"]).drop_duplicates()
print(f"{len(df_refino)} correções; por estratégia: {df_refino['estratégia'].value_counts().to_dict()}")
mostrar(df_refino, n=40, largura=60)

## 8. Grafo normalizado e relações — *normalizacao*
As menções refinadas entram num `GraphBuilder` novo, que expande siglas, normaliza o rótulo e deduplica por
atributos de identidade (sintoma presente e ausente continuam nós distintos). A evidência de cada menção é
aparada para que `case_text[char_start:char_end] == evidence_text`, e `build_relations` cria as arestas.

In [ ]:
impacto = final.normalization_impact()
print(f"{len(final.nodes)} nós e {len(final.edges)} arestas")
print("impacto da normalização:", impacto.to_dict())
print("nós por tipo:", dict(Counter(n.type for n in final.nodes)))
print("arestas por relação:", dict(Counter(e.relation for e in final.edges)))

## 9. Sítios anatômicos — *dicionarios*
O gazetteer anatômico curado (`anatomical_site_gazetteer.csv`) é casado contra os tokens do caso.
Como em dicionarios, o órgão embutido num diagnóstico ou achado (`gastric duplication cyst`) não vira sítio.
Cada sítio se liga por `LOCATED_IN` ao `Symptom`/`Finding`/`Treatment` mais próximo na mesma sentença.

In [ ]:
linhas = dados["sitios"]
print(f"{len(linhas)} ocorrências de termos anatômicos")
mostrar(pd.DataFrame(linhas), n=30, largura=70)

## 10. Conceitos MeSH — *dicionarios*
O gazetteer MeSH é carregado uma vez e cada tipo consulta só a sua categoria (mesma rota de `dicionarios/pipeline.py`).
Como os rótulos já estão curtos, um casamento parcial só é aceito se cobrir o último token (o núcleo):
`right flank lower quadrant abdominal pain` liga a *Abdominal Pain*, mas um rótulo não liga a um termo solto do meio.
O casamento aproximado (rapidfuzz, limiar 85, mínimo de 5 caracteres) só roda quando nada casa exatamente.

In [ ]:
df_mesh = pd.DataFrame(dados["mesh"], columns=["node_id", "type", "label", "categoria", "code", "preferred_term", "estratégia", "score"])
ligados = int((df_mesh["code"] != "").sum())
print(f"{ligados} de {len(df_mesh)} nós ligáveis ganharam SAME_AS")
mostrar(df_mesh, n=40, largura=45)

## 11. Tabelas finais
Nós e arestas no contrato comum. O rótulo de `ExamResult` é refeito a partir de `value` e `unit`, para usar a unidade canônica
(`ng/mL`) em vez da forma minúscula que a normalização de texto produz. As verificações conferem os offsets de toda evidência e o
domínio de cada relação conforme `docs/01-dados-a-extrair.md` §2 e `docs/02-esquema-grafo.md`.

In [ ]:
nos, arestas = resultado.nos, resultado.arestas
validacao = validar_grafo(caso, resultado)
print("Evidências desalinhadas:", validacao.evidencias_desalinhadas)
print("Arestas fora do domínio:", validacao.arestas_fora_do_dominio)
print(f"Nós ({len(nos)}):", nos["type"].value_counts().to_dict())
mostrar(nos[["node_id", "type", "label", "attributes"]], n=60, largura=80)


In [ ]:
rotulo_de = {n.node_id: rotulo_final(n) for n in final.nodes}
vista = pd.DataFrame(
    [
        {
            "edge_id": e.edge_id,
            "origem": f"{e.source_id} {curto(rotulo_de[e.source_id], 28)}",
            "relation": e.relation,
            "destino": f"{e.target_id} {curto(rotulo_de[e.target_id], 28)}",
            "trigger / estratégia": e.attributes.get("trigger", e.attributes.get("strategy")),
            "evidência": curto(e.attributes.get("evidence_text", ""), 50),
        }
        for e in final.edges
    ]
)
print(f"Arestas ({len(arestas)}):", arestas["relation"].value_counts().to_dict())
mostrar(vista, n=80, largura=60)

Atributos expandidos, um bloco por tipo de entidade:

In [ ]:
for tipo, grupo in Counter(n.type for n in final.nodes).items():
    linhas = [{"node_id": n.node_id, "label": curto(rotulo_final(n), 40), **n.attributes} for n in final.nodes if n.type == tipo]
    df_tipo = pd.DataFrame(linhas).dropna(axis=1, how="all")
    print(f"\n{tipo} ({grupo})")
    mostrar(df_tipo, n=12, largura=35)

## 12. Exportação
As duas tabelas vão para `data/processed/<case_id>-nodes.csv` e `-edges.csv`.

In [ ]:
SAIDA = ROOT / "data" / "processed"
SAIDA.mkdir(parents=True, exist_ok=True)
caminho_nos = SAIDA / f"{CASE_ID}-nodes.csv"
caminho_arestas = SAIDA / f"{CASE_ID}-edges.csv"
nos.to_csv(caminho_nos, index=False)
arestas.to_csv(caminho_arestas, index=False)
print(f"{caminho_nos.relative_to(ROOT)}: {len(nos)} linhas")
print(f"{caminho_arestas.relative_to(ROOT)}: {len(arestas)} linhas")